# MarsLandformNet V3 — Tile Classifier GPU Training (V2 Fixed)

**Setup:** Runtime → Change runtime type → **GPU (T4)**

**Key fixes from v1 notebook (which got F1=0.03):**
1. Simplified loss: weighted CE only (removed focal, label smoothing, KL-div)
2. Removed double-boosting: class_weights in CE only (no WeightedRandomSampler)
3. LR 1e-3 with ReduceLROnPlateau (was OneCycleLR at 1e-4)
4. Pure 768-dim input (dropped zero-MOLA)
5. Added BatchNorm, lower dropout (0.1)
6. Smaller batch (128) for more gradient steps
7. Added overfit sanity check cell
8. Prediction distribution logging per epoch

Run all cells top-to-bottom.

In [ ]:
# Cell 1: Download data
import subprocess, os
from pathlib import Path

ROOT = Path('/content/marslab_v3')
ROOT.mkdir(exist_ok=True)
os.chdir(ROOT)

TAR = ROOT / 'v3_training_data.tar.gz'
if not TAR.exists():
    URL = 'https://github.com/jejuchild/MarsLab/releases/download/v3-training-data/v3_training_data.tar.gz'
    print(f'Downloading from {URL}...')
    subprocess.check_call(['wget', '-q', '-O', str(TAR), URL])
    print('Downloaded!')

subprocess.check_call(['tar', 'xzf', str(TAR), '-C', str(ROOT)])
print('Extracted to', ROOT)
!find {ROOT}/Data -type f | head -10

In [ ]:
# Cell 2: Install deps + check GPU
!pip install -q numpy torch torchvision scikit-learn
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Cell 3: Model + Dataset definitions
import json
import logging
import time
from collections import Counter
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('v3_train')

V3_CLASSES = ['LDA', 'LVF', 'CCF', 'OTHER']
ROOT = Path('/content/marslab_v3')


@dataclass
class TileClassifierConfig:
    embed_dim: int = 768
    hidden_dim: int = 256
    num_classes: int = 4
    dropout: float = 0.1
    lr: float = 1e-3
    weight_decay: float = 1e-4
    epochs: int = 150
    patience: int = 25
    batch_size: int = 128
    other_subsample_ratio: float = 0.3


class TileLandformClassifier(nn.Module):
    """embedding (768) -> MLP -> 4 classes."""
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.classifier = nn.Sequential(
            nn.Linear(config.embed_dim, config.hidden_dim),
            nn.BatchNorm1d(config.hidden_dim),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_dim, config.hidden_dim),
            nn.BatchNorm1d(config.hidden_dim),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_dim, config.num_classes),
        )

    def forward(self, embeddings):
        return self.classifier(embeddings)


class TileLabelDataset(Dataset):
    def __init__(self, tile_labels, tile_indices, config, is_train=True, embeddings_by_image=None):
        self.config = config
        self.is_train = is_train
        self.embeddings_by_image = embeddings_by_image
        self.samples = []
        other_samples = []
        self.class_to_idx = {cls: i for i, cls in enumerate(V3_CLASSES)}
        for idx in tile_indices:
            t = tile_labels[idx]
            label = t.get('label')
            if label == 'UNLABELED' or label is None:
                continue
            img_id = t['image_id']
            if embeddings_by_image is not None and img_id not in embeddings_by_image:
                continue
            sample = {'image_id': img_id, 'tile_idx': t['tile_idx'], 'label': label}
            if label == 'OTHER':
                other_samples.append(sample)
            else:
                self.samples.append(sample)
        if is_train and other_samples:
            import random
            n_keep = max(1, int(len(other_samples) * config.other_subsample_ratio))
            random.shuffle(other_samples)
            other_samples = other_samples[:n_keep]
        self.samples.extend(other_samples)
        logger.info('Dataset: %d samples %s', len(self.samples), dict(Counter(s['label'] for s in self.samples)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        all_emb = self.embeddings_by_image[sample['image_id']]
        ti = sample['tile_idx']
        embedding = all_emb[ti] if ti < len(all_emb) else np.zeros((self.config.embed_dim,), dtype=np.float32)
        label_idx = self.class_to_idx.get(sample['label'], self.class_to_idx['OTHER'])
        return {
            'embedding': torch.from_numpy(embedding).float(),
            'label': torch.tensor(label_idx, dtype=torch.long),
        }

    def get_class_weights(self):
        """Inverse-frequency weights, sqrt-dampened to avoid extreme ratios."""
        counts = Counter(s['label'] for s in self.samples)
        total = len(self.samples)
        weights = torch.zeros(len(V3_CLASSES))
        for cls, idx in self.class_to_idx.items():
            freq = max(counts.get(cls, 1), 1) / total
            weights[idx] = 1.0 / np.sqrt(freq)  # sqrt-dampened
        # Normalize so mean weight = 1
        weights = weights / weights.mean()
        return weights


def _collate(batch):
    return {
        'embedding': torch.stack([b['embedding'] for b in batch]),
        'label': torch.stack([b['label'] for b in batch]),
    }


print('Model code loaded.')

In [ ]:
# Cell 4: Load data
LABELS_PATH = ROOT / 'Data/HiRISE/v3_output/tile_labels_v3.json'
SPLITS_PATH = ROOT / 'Data/HiRISE/v3_output/tile_splits_v3.json'
EMB_PATH = ROOT / 'Data/HiRISE/v2_output/embeddings_ssl/embeddings_by_image.npy'

with open(LABELS_PATH) as f:
    tile_labels = json.load(f)
with open(SPLITS_PATH) as f:
    splits = json.load(f)

emb = np.load(str(EMB_PATH), allow_pickle=True).item()
print(f'Loaded {len(tile_labels)} tile labels, {len(emb)} image embeddings')
print(f'Splits: train={len(splits["train"])}, val={len(splits["val"])}, test={len(splits["test"])}')

# Sanity check embeddings
sample_key = list(emb.keys())[0]
sample_emb = emb[sample_key]
print(f'\nEmb check: {sample_key} shape={sample_emb.shape}, mean={sample_emb.mean():.4f}, std={sample_emb.std():.4f}')
assert sample_emb.shape[1] == 768, f'Expected 768-dim, got {sample_emb.shape[1]}'

cfg = TileClassifierConfig()
train_ds = TileLabelDataset(tile_labels, splits['train'], cfg, True, emb)
val_ds = TileLabelDataset(tile_labels, splits['val'], cfg, False, emb)
test_ds = TileLabelDataset(tile_labels, splits['test'], cfg, False, emb)

print(f'\nEffective: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}')

# Verify embedding-label alignment: spot-check 5 samples
print('\n--- Alignment check (5 samples) ---')
for i in [0, 100, 500, 1000, len(train_ds)-1]:
    s = train_ds.samples[i]
    item = train_ds[i]
    emb_norm = item['embedding'].norm().item()
    print(f'  [{i}] img={s["image_id"]} tile={s["tile_idx"]} label={s["label"]} emb_norm={emb_norm:.2f} all_zero={emb_norm < 0.01}')

In [ ]:
# Cell 5: OVERFIT SANITY CHECK
# If this can't reach >90% train accuracy on 512 samples in 50 steps,
# there's a data/wiring bug (not a hyperparameter issue).
print('=== Overfit Sanity Check ===')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Take first 512 training samples
sanity_ds = torch.utils.data.Subset(train_ds, list(range(min(512, len(train_ds)))))
sanity_loader = DataLoader(sanity_ds, batch_size=128, shuffle=True, collate_fn=_collate)

sanity_model = TileLandformClassifier(cfg).to(device)
sanity_opt = torch.optim.Adam(sanity_model.parameters(), lr=1e-3)
sanity_loss_fn = nn.CrossEntropyLoss()  # no weights, no tricks

for step in range(50):
    sanity_model.train()
    total_correct, total_samples, total_loss = 0, 0, 0.0
    for batch in sanity_loader:
        e = batch['embedding'].to(device)
        y = batch['label'].to(device)
        logits = sanity_model(e)
        loss = sanity_loss_fn(logits, y)
        sanity_opt.zero_grad()
        loss.backward()
        sanity_opt.step()
        total_correct += (logits.argmax(1) == y).sum().item()
        total_samples += len(y)
        total_loss += loss.item()
    acc = total_correct / total_samples
    if (step + 1) % 10 == 0 or step == 0:
        pred_dist = Counter(logits.argmax(1).cpu().tolist())
        print(f'  Step {step+1:3d}  loss={total_loss:.4f}  train_acc={acc:.4f}  preds={dict(pred_dist)}')
    if acc > 0.95:
        print(f'  ✓ Overfit check PASSED at step {step+1} (acc={acc:.4f})')
        break
else:
    if acc < 0.5:
        print(f'  ✗ WARNING: Could not overfit! acc={acc:.4f} — possible data/wiring bug!')
    else:
        print(f'  ~ Partial overfit: acc={acc:.4f} (model is learning but slowly)')

del sanity_model, sanity_opt  # free memory
print()

In [ ]:
# Cell 6: Full Training
model = TileLandformClassifier(cfg).to(device)
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

# Class-weighted CE only (NO WeightedRandomSampler — avoid double-boosting)
class_weights = train_ds.get_class_weights().to(device)
print(f'Class weights: {dict(zip(V3_CLASSES, [f"{w:.3f}" for w in class_weights.tolist()]))}')
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=7, min_lr=1e-6
)

# Regular shuffle (no weighted sampler — class_weights in CE handles imbalance)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=2, pin_memory=True, collate_fn=_collate)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size * 2, shuffle=False,
                        num_workers=2, pin_memory=True, collate_fn=_collate)

from sklearn.metrics import f1_score, accuracy_score, classification_report

best_f1 = 0.0
best_epoch = -1
patience_counter = 0
SAVE_PATH = ROOT / 'best_tile_classifier.pt'

print(f'\nBatches/epoch: {len(train_loader)}')
print(f'Batch size: {cfg.batch_size}, LR: {cfg.lr}, Dropout: {cfg.dropout}')
print(f'Patience: {cfg.patience}, Max epochs: {cfg.epochs}')
print('\n--- Starting Training ---\n')

for epoch in range(1, cfg.epochs + 1):
    t0 = time.time()
    model.train()
    train_loss_sum, n_train = 0.0, 0
    train_correct, train_total = 0, 0
    for batch in train_loader:
        emb_b = batch['embedding'].to(device)
        labels_b = batch['label'].to(device)
        logits = model(emb_b)
        loss = criterion(logits, labels_b)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss_sum += loss.item()
        n_train += 1
        train_correct += (logits.argmax(1) == labels_b).sum().item()
        train_total += len(labels_b)

    train_acc = train_correct / max(train_total, 1)

    # Validation
    model.eval()
    all_preds, all_labels_list = [], []
    val_loss_sum, n_val = 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            emb_b = batch['embedding'].to(device)
            labels_b = batch['label'].to(device)
            logits = model(emb_b)
            val_loss_sum += criterion(logits, labels_b).item()
            n_val += 1
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_labels_list.extend(labels_b.cpu().tolist())

    # Metrics
    lf_mask = [i for i, y in enumerate(all_labels_list) if y < 3]
    lf_f1 = f1_score([all_labels_list[i] for i in lf_mask], [all_preds[i] for i in lf_mask],
                     average='macro', zero_division=0) if lf_mask else 0.0
    overall_f1 = f1_score(all_labels_list, all_preds, average='macro', zero_division=0)
    acc = accuracy_score(all_labels_list, all_preds)
    elapsed = time.time() - t0

    # Prediction distribution
    pred_dist = Counter(all_preds)
    pred_str = ' '.join(f'{V3_CLASSES[k]}={v}' for k, v in sorted(pred_dist.items()))

    tl = train_loss_sum / max(n_train, 1)
    vl = val_loss_sum / max(n_val, 1)
    lr_now = optimizer.param_groups[0]['lr']
    print(f'Ep {epoch:3d}/{cfg.epochs}  tl={tl:.4f} vl={vl:.4f}  t_acc={train_acc:.3f} v_acc={acc:.3f}  lf_F1={lf_f1:.4f} all_F1={overall_f1:.4f}  lr={lr_now:.1e}  [{pred_str}]  ({elapsed:.1f}s)')

    scheduler.step(lf_f1)

    # Full report every 10 epochs
    if epoch % 10 == 0 or epoch == 1:
        print(classification_report(all_labels_list, all_preds,
              target_names=V3_CLASSES, zero_division=0, digits=4))

    if lf_f1 > best_f1:
        best_f1 = lf_f1
        best_epoch = epoch
        patience_counter = 0
        torch.save({'model_state_dict': model.state_dict(), 'config': asdict(cfg),
                     'epoch': epoch, 'best_landform_macro_f1': lf_f1,
                     'overall_macro_f1': overall_f1,
                     'classes': V3_CLASSES}, SAVE_PATH)
        print(f'  >> Saved (lf_F1={lf_f1:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= cfg.patience:
            print(f'Early stopping at epoch {epoch} (patience={cfg.patience})')
            break

print(f'\nDone. Best landform F1={best_f1:.4f} at epoch {best_epoch}')

In [ ]:
# Cell 7: Evaluate best checkpoint on val + test
ckpt = torch.load(SAVE_PATH, map_location='cpu')
eval_model = TileLandformClassifier(cfg)
eval_model.load_state_dict(ckpt['model_state_dict'])
eval_model.eval()

def evaluate_split(ds, name):
    loader = DataLoader(ds, batch_size=1024, shuffle=False, collate_fn=_collate)
    yt, yp = [], []
    with torch.no_grad():
        for batch in loader:
            logits = eval_model(batch['embedding'])
            yp.extend(logits.argmax(dim=1).tolist())
            yt.extend(batch['label'].tolist())
    overall = f1_score(yt, yp, average='macro', zero_division=0)
    lf_idx = [i for i, y in enumerate(yt) if y < 3]
    lf_f1 = f1_score([yt[i] for i in lf_idx], [yp[i] for i in lf_idx],
                     average='macro', zero_division=0) if lf_idx else 0.0
    print(f'\n=== {name} ({len(ds)} samples) ===')
    print(f'  Overall macro-F1: {overall:.4f}')
    print(f'  Landform macro-F1 (LDA/LVF/CCF): {lf_f1:.4f}')
    print(classification_report(yt, yp, target_names=V3_CLASSES, zero_division=0, digits=4))
    per = {}
    for c, cls in enumerate(V3_CLASSES):
        per[cls] = round(f1_score([1 if y==c else 0 for y in yt],
                                  [1 if p==c else 0 for p in yp], zero_division=0), 4)
    return {'samples': len(ds), 'overall_f1': overall, 'landform_f1': lf_f1, 'per_class': per}

val_result = evaluate_split(val_ds, 'Validation')
test_result = evaluate_split(test_ds, 'Test')

In [ ]:
# Cell 8: Save to Google Drive
from google.colab import drive
import shutil

drive.mount('/content/drive')
drive_dir = Path('/content/drive/MyDrive/MarsLab_V3')
drive_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(SAVE_PATH, drive_dir / 'best_tile_classifier.pt')

summary = {'best_f1': best_f1, 'best_epoch': best_epoch,
           'config': asdict(cfg), 'val': val_result, 'test': test_result}
with open(drive_dir / 'training_results.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Saved to Google Drive: {drive_dir}')
print('  best_tile_classifier.pt')
print('  training_results.json')